In [13]:

import os
import sys
import types
import math
import random

import numpy as np
import networkx as nx
import torch
import torch.nn as nn

from build_graph_data import build_cordon_graph_data
from model import PolicyAwareHeuristicNet

In [2]:

# -------------------------------------------------------
# User-provided grid generator (kept almost unchanged)
# -------------------------------------------------------
def build_grid_graph_with_pos(n_rows=4, n_cols=4): 
    G = nx.grid_2d_graph(n_rows, n_cols)
    G = nx.convert_node_labels_to_integers(G, ordering="sorted")  # node ids 0..N-1
    pos_map = {}
    for idx, (i, j) in enumerate([(i, j) for i in range(n_rows) for j in range(n_cols)]):
        pos_map[idx] = (j, -i)   # x=j, y=-i
        G.nodes[idx]['pos'] = (j, -i)

    # user-provided edge attributes
    for u, v in G.edges():
        G.edges[u, v]['free_time'] = float(np.random.uniform(10, 50))
        G.edges[u, v]['capacity'] = float(np.random.uniform(100, 500))
    return G, pos_map


# -------------------------------------------------------
# Convert grid graph to links format required by our code:
# (u, v, length, free_time, capacity)
# -------------------------------------------------------
def graph_to_links(G, pos_map):
    links = []
    for u, v, data in G.edges(data=True):
        x1, y1 = pos_map[u]
        x2, y2 = pos_map[v]
        length = float(((x1 - x2) ** 2 + (y1 - y2) ** 2) ** 0.5)
        free_time = float(data['free_time'])
        capacity = float(data['capacity'])
        links.append((int(u), int(v), length, free_time, capacity))
    return links


# -------------------------------------------------------
# Build a synthetic OD matrix in dict format:
# {(o, d): demand}
# -------------------------------------------------------
def build_synthetic_od(G, n_pairs=20, demand_low=50, demand_high=300, seed=7):
    rng = np.random.default_rng(seed)
    nodes = list(G.nodes())
    od = {}
    sampled = set()
    while len(sampled) < n_pairs:
        o, d = rng.choice(nodes, size=2, replace=False)
        pair = (int(o), int(d))
        if pair in sampled:
            continue
        sampled.add(pair)
        od[pair] = float(rng.integers(demand_low, demand_high))
    return od


In [5]:

# Build synthetic road network and OD demand
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

G, pos_map = build_grid_graph_with_pos(n_rows=4, n_cols=4)
links = graph_to_links(G, pos_map)
od_demand = build_synthetic_od(G, n_pairs=18, seed=42)

print(f"#real nodes = {G.number_of_nodes()}")
print(f"#real undirected edges = {G.number_of_edges()}")
print(f"#links for MSA = {len(links)}")
print(f"#OD pairs = {len(od_demand)}")
print("sample links:")
for x in links[:5]:
    print("  ", x)
print("sample OD pairs:")
for i, kv in enumerate(od_demand.items()):
    if i >= 5:
        break
    print("  ", kv)


#real nodes = 16
#real undirected edges = 24
#links for MSA = 24
#OD pairs = 18
sample links:
   (0, 4, 1.0, 24.9816047538945, 480.2857225639665)
   (0, 1, 1.0, 39.2797576724562, 339.46339367881467)
   (1, 5, 1.0, 16.24074561769746, 162.39780813448107)
   (1, 2, 1.0, 12.32334448672798, 446.47045830997405)
   (2, 6, 1.0, 34.04460046972835, 383.2290311184182)
sample OD pairs:
   ((1, 12), 159.0)
   ((13, 6), 224.0)
   ((3, 1), 293.0)
   ((11, 12), 246.0)
   ((7, 2), 162.0)


In [6]:

# Build data package for toll policy
pack_toll = build_cordon_graph_data(
    links=links,
    od_demand=od_demand,
    policy_type="toll",
    virtual_node=-1,
)

# Build data package for speed-limit policy
pack_speed = build_cordon_graph_data(
    links=links,
    od_demand=od_demand,
    policy_type="speed_limit",
    virtual_node=-1,
)

print("Built both toll and speed-limit packs successfully.")


Built both toll and speed-limit packs successfully.


In [7]:

# -------------------------------------------------------
# Check outputs for MSA and environment
# -------------------------------------------------------
print("Keys in returned pack:", list(pack_toll.keys()))
print("MSA links length:", len(pack_toll["links"]))
print("MSA od_demand length:", len(pack_toll["od_demand"]))
print("road_graph nodes/edges:", pack_toll["road_graph"].number_of_nodes(), pack_toll["road_graph"].number_of_edges())
print("virtual node in road_graph?", -1 in pack_toll["road_graph"].nodes)


Keys in returned pack: ['links', 'od_demand', 'road_graph', 'model_data', 'node2idx', 'idx2node']
MSA links length: 24
MSA od_demand length: 18
road_graph nodes/edges: 16 24
virtual node in road_graph? False


In [16]:
pack_toll['model_data'].policy_type_id

tensor(0)

In [8]:

# -------------------------------------------------------
# Check model input tensors
# -------------------------------------------------------
data = pack_toll["model_data"]
print("num model nodes =", data.num_nodes)
print("node_ids[:8] =", data.node_ids[:8])
print("virtual node index =", data.node2idx[-1])
print("road_x shape =", tuple(data.road_x.shape))
print("road_edge_index shape =", tuple(data.road_edge_index.shape))
print("road_edge_attr shape =", tuple(data.road_edge_attr.shape))
print("od_x shape =", tuple(data.od_x.shape))
print("od_edge_index shape =", tuple(data.od_edge_index.shape))
print("od_edge_attr shape =", tuple(data.od_edge_attr.shape))
print("pair_feat shape =", tuple(data.pair_feat.shape))
print("policy_type_id (toll) =", int(data.policy_type_id.item()))
print("policy_type_id (speed_limit) =", int(pack_speed["model_data"].policy_type_id.item()))


num model nodes = 17
node_ids[:8] = [-1, 0, 1, 2, 3, 4, 5, 6]
virtual node index = 0
road_x shape = (17, 1)
road_edge_index shape = (2, 80)
road_edge_attr shape = (80, 3)
od_x shape = (17, 1)
od_edge_index shape = (2, 50)
od_edge_attr shape = (50, 1)
pair_feat shape = (17, 17, 4)
policy_type_id (toll) = 0
policy_type_id (speed_limit) = 1


In [9]:

# -------------------------------------------------------
# Verify virtual-node features were encoded as requested
# -------------------------------------------------------
vidx = data.node2idx[-1]

# Road edges involving virtual node should have edge_attr = [-1, -1, -1]
road_src = data.road_edge_index[0]
road_dst = data.road_edge_index[1]
mask_road_virtual = (road_src == vidx) | (road_dst == vidx)
virtual_road_attr = data.road_edge_attr[mask_road_virtual]
print("#road edges touching virtual node =", int(mask_road_virtual.sum().item()))
print("unique virtual road edge rows:")
print(torch.unique(virtual_road_attr, dim=0))

# OD edges involving virtual node should have demand = -1
od_src = data.od_edge_index[0]
od_dst = data.od_edge_index[1]
mask_od_virtual = (od_src == vidx) | (od_dst == vidx)
virtual_od_attr = data.od_edge_attr[mask_od_virtual]
print("#od edges touching virtual node =", int(mask_od_virtual.sum().item()))
print("unique virtual od edge rows:")
print(torch.unique(virtual_od_attr, dim=0))


#road edges touching virtual node = 32
unique virtual road edge rows:
tensor([[-1., -1., -1.]])
#od edges touching virtual node = 32
unique virtual od edge rows:
tensor([[-1.]])


In [10]:

# -------------------------------------------------------
# Forward pass through heuristic model
# -------------------------------------------------------
model_toll = PolicyAwareHeuristicNet(hidden_dim=32, road_layers=1, od_layers=1, heads=4)
model_speed = PolicyAwareHeuristicNet(hidden_dim=32, road_layers=1, od_layers=1, heads=4)

with torch.no_grad():
    heu_toll = model_toll(pack_toll["model_data"])
    heu_speed = model_speed(pack_speed["model_data"])

print("heu_toll shape =", tuple(heu_toll.shape))
print("heu_speed shape =", tuple(heu_speed.shape))
print("heu_toll min/max =", float(heu_toll.min()), float(heu_toll.max()))
print("heu_speed min/max =", float(heu_speed.min()), float(heu_speed.max()))


heu_toll shape = (17, 17)
heu_speed shape = (17, 17)
heu_toll min/max = 9.99999993922529e-09 25.25686264038086
heu_speed min/max = 9.99999993922529e-09 0.7088981866836548


In [11]:

# A few sanity checks
assert heu_toll.shape[0] == heu_toll.shape[1] == pack_toll["model_data"].num_nodes
assert heu_speed.shape[0] == heu_speed.shape[1] == pack_speed["model_data"].num_nodes
assert torch.all(torch.isfinite(heu_toll))
assert torch.all(torch.isfinite(heu_speed))
assert float(heu_toll.min()) >= 0.0
assert float(heu_speed.min()) >= 0.0

print("All forward-pass checks passed.")


All forward-pass checks passed.


In [12]:

# Inspect a small top-left block of the heuristic matrix
print("Top-left 6x6 block of heu_toll:")
print(heu_toll[:6, :6])

print("Top-left 6x6 block of heu_speed:")
print(heu_speed[:6, :6])


Top-left 6x6 block of heu_toll:
tensor([[1.0000e-08, 8.4664e-01, 8.4671e-01, 8.4670e-01, 8.4664e-01, 8.4636e-01],
        [8.4974e-01, 1.0000e-08, 1.8100e+01, 7.9500e-01, 7.9532e-01, 2.5071e+01],
        [8.4933e-01, 1.8100e+01, 1.0000e-08, 2.3163e+01, 7.9519e-01, 7.9490e-01],
        [8.4932e-01, 7.9519e-01, 2.3163e+01, 1.0000e-08, 2.5257e+01, 7.9489e-01],
        [8.4975e-01, 7.9533e-01, 9.1110e+00, 2.5257e+01, 1.0000e-08, 7.9502e-01],
        [8.4953e-01, 2.5071e+01, 7.9494e-01, 7.9494e-01, 7.9527e-01, 1.0000e-08]])
Top-left 6x6 block of heu_speed:
tensor([[1.0000e-08, 6.4463e-01, 6.4495e-01, 6.4495e-01, 6.4463e-01, 6.4469e-01],
        [6.3268e-01, 1.0000e-08, 1.0002e-08, 7.0884e-01, 7.0828e-01, 1.0000e-08],
        [6.3226e-01, 1.0002e-08, 1.0000e-08, 1.0000e-08, 7.0826e-01, 7.0833e-01],
        [6.3226e-01, 7.0827e-01, 1.0000e-08, 1.0000e-08, 1.0000e-08, 7.0833e-01],
        [6.3268e-01, 7.0828e-01, 1.0005e-08, 1.0000e-08, 1.0000e-08, 7.0834e-01],
        [6.3261e-01, 1.0000e-08,